## Wczytanie danych

In [109]:
import pandas as pd
import numpy as np

df = pd.read_csv('data.csv')
df = df.drop(columns=['instant', 'dteday', 'casual', 'registered'])
df['cnt'] = np.log1p(df['cnt'])

## Normalizacja danych

In [110]:
train = df.sample(frac=0.7, random_state=200)
remaining = df.drop(train.index)
val = remaining.sample(frac=0.5, random_state=200)
test = remaining.drop(val.index)

mean = train.iloc[:,:-1].mean()
std = train.iloc[:,:-1].std()

train_norm = (train.iloc[:,:-1] - mean) / std
val_norm = (val.iloc[:,:-1] - mean) / std
test_norm = (test.iloc[:,:-1] - mean) / std


## Utworzenie datasetów

In [111]:
import torch.utils.data as data
import torch.nn as nn
import torch

train_dataset = data.TensorDataset(torch.from_numpy(train_norm.values).float(),torch.from_numpy(train.values[:,-1]).float())
val_dataset = data.TensorDataset(torch.from_numpy(val_norm.values).float(), torch.from_numpy(val.values[:,-1]).float())
test_dataset = data.TensorDataset(torch.from_numpy(test_norm.values).float(), torch.from_numpy(test.values[:,-1]).float())

## Batchowanie danych treningowych

In [112]:
train_data_loader = data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_data_loader = data.DataLoader(val_dataset, batch_size=64, shuffle=False)
test_data_loader = data.DataLoader(test_dataset, batch_size=64, shuffle=False, drop_last=False)

## Struktura modelu

In [ ]:
class Model(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_outputs):
        super().__init__()

        self.linear1 = nn.Linear(n_inputs, n_hidden1)
        self.linear2 = nn.Linear(n_hidden1, n_hidden2)
        self.linear3 = nn.Linear(n_hidden2, n_outputs)

        self.act_fn = nn.ReLU()
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.act_fn(x)

        x = self.linear2(x)
        x = self.act_fn(x)

        x = self.linear3(x)

        return x

In [ ]:
model = Model(train_norm.shape[1], 32, 16, 1)
device = torch.device('mps')
model.to(device)

Model(
  (linear1): Linear(in_features=12, out_features=32, bias=True)
  (linear2): Linear(in_features=32, out_features=16, bias=True)
  (linear3): Linear(in_features=16, out_features=1, bias=True)
  (act_fn): ReLU()
)

In [115]:
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
loss_module = nn.MSELoss()

def rmsle(y_true,y_pred):
    n = len(y_true)
    msle = np.mean([(np.log(max(y_pred[i],0) + 1) - np.log(y_true[i] + 1)) ** 2.0 for i in range(n)])
    return np.sqrt(msle)

In [116]:
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data_inputs, data_labels in dataloader:
            data_inputs = data_inputs.to(device)
            data_labels = data_labels.to(device)

            preds = model(data_inputs)
            preds = preds.squeeze(dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(data_labels.cpu().numpy())
            
            loss = loss_module(preds, data_labels)
            total_loss += loss.item()
            
    mse = total_loss/len(dataloader)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    rmsle_val = rmsle(all_labels, all_preds)

    print(f"Loss: {mse:.3}, RMSLE: {rmsle_val}")

In [117]:
def train(model, epochs, train_data_loader, val_data_loader):
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for data_inputs, data_labels in train_data_loader:
            data_inputs = data_inputs.to(device)
            data_labels = data_labels.to(device)

            preds = model(data_inputs)
            preds = preds.squeeze(dim=1)

            loss = loss_module(preds, data_labels)

            optimizer.zero_grad()
            loss.backward()

            optimizer.step()
            total_loss += loss.item()
        
        if epoch % 10 == 0:
            evaluate(model, val_data_loader)
            model.train()

In [118]:
train(model, 300, train_data_loader, val_data_loader)

Loss: 2.43, RMSLE: 0.32877588272094727
Loss: 1.03, RMSLE: 0.23651285469532013
Loss: 0.561, RMSLE: 0.18956567347049713
Loss: 0.32, RMSLE: 0.14528228342533112
Loss: 0.194, RMSLE: 0.1131816878914833
Loss: 0.148, RMSLE: 0.0971916988492012
Loss: 0.132, RMSLE: 0.09342668950557709
Loss: 0.131, RMSLE: 0.09618140012025833
Loss: 0.123, RMSLE: 0.08822959661483765
Loss: 0.126, RMSLE: 0.0940796434879303
Loss: 0.112, RMSLE: 0.08742158114910126
Loss: 0.11, RMSLE: 0.08745259791612625
Loss: 0.118, RMSLE: 0.08858022093772888
Loss: 0.111, RMSLE: 0.0890246033668518
Loss: 0.104, RMSLE: 0.08427944034337997
Loss: 0.103, RMSLE: 0.08510835468769073
Loss: 0.108, RMSLE: 0.08777247369289398
Loss: 0.101, RMSLE: 0.08466365188360214
Loss: 0.103, RMSLE: 0.0842544436454773
Loss: 0.11, RMSLE: 0.0862000584602356
Loss: 0.105, RMSLE: 0.0851663202047348
Loss: 0.102, RMSLE: 0.08339816331863403
Loss: 0.0996, RMSLE: 0.0832667127251625
Loss: 0.104, RMSLE: 0.08412902057170868
Loss: 0.102, RMSLE: 0.083645299077034
Loss: 0.103, R

In [119]:
evaluate(model, test_data_loader)

Loss: 0.1, RMSLE: 0.08315455913543701


In [124]:
eval_df = pd.read_csv('evaluation_data.csv')
eval_features = eval_df.drop(columns=['dteday'])
eval_norm = (eval_features - mean) / std

eval_tensor = torch.from_numpy(eval_norm.values).float().to(device)

model.eval()
with torch.no_grad():
    preds = model(eval_tensor)
    preds = preds.squeeze(dim=1)

    preds = np.expm1(preds.cpu().numpy())

submission = pd.DataFrame(preds, columns=['cnt'])
submission.to_csv('sroda_Pędziwiatr_Siemionek.csv', index=False)